# Project 4 Live Demo：推論 + PD 控制（可切換模型）

流程：`camera.py` 取像 → `model.py` 推論 → `control.py` PD + bias 差速控制。

- 上方下拉選單切換 checkpoint（不同 augmentation / 有無 DepGraph 剪枝 / PyTorch 或 TensorRT）。
- 「允許移動」**預設關閉**：關閉時只推論、顯示，不送馬達。第一次請把車架高或抓好。
- 「輸入干擾」可以在即時畫面上加柏林噪聲 / 壓縮失真，測各模型的穩健性。
- 「停止」會立刻停車並取消移動許可；切換模型時也會自動停車並取消移動許可。

`models/` 放 `train.py` 產生的 `*.pth`（及同名 `.json`），以及板子上用 `trtexec` + `compression.wrap_trt_engine` 做出的 TRT `.pth`。

## 1. 設定

In [ ]:
import os, sys, time, threading, csv, datetime
import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, os.path.abspath('.'))   # project4/ 內的模組
import camera as cam_mod
from model import load_model, list_models
from control import PDController, model_output_to_xy

MODELS_DIR = 'models'
CAMERA_SOURCE = 'jetbot'       # 'jetbot' | 'opencv' | 'folder' | 'dummy'
CAMERA_FOLDER = 'data/dataset_xy_2026-10-21/dataset_xy'   # CAMERA_SOURCE='folder' 時使用
USE_ROBOT = True               # False：沒有 JetBot 也能跑（馬達值只顯示）
LOG_DIR = 'runs'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| models:', [os.path.basename(p) for _, p in list_models(MODELS_DIR)])

## 2. 相機與車子

In [ ]:
camera = cam_mod.open_camera(CAMERA_SOURCE, folder=CAMERA_FOLDER)

class _FakeMotor(object):
    value = 0.0

class _FakeRobot(object):
    def __init__(self):
        self.left_motor, self.right_motor = _FakeMotor(), _FakeMotor()
    def stop(self):
        self.left_motor.value = self.right_motor.value = 0.0

if USE_ROBOT:
    from jetbot import Robot
    robot = Robot()
else:
    robot = _FakeRobot()
robot.stop()
controller = PDController()

## 3. 介面

In [ ]:
S = dict(layout=widgets.Layout(width='320px'), style={'description_width': '110px'})
R = dict(disabled=True, readout_format='.3f', **S)

# --- 變因 ---------------------------------------------------------------
model_dd = widgets.Dropdown(description='模型', options=list_models(MODELS_DIR),
                            layout=widgets.Layout(width='520px'), style={'description_width': '50px'})
refresh_btn = widgets.Button(description='重新掃描', layout=widgets.Layout(width='90px'))
model_info = widgets.HTML(value='(尚未載入)')
noise_perlin = widgets.Checkbox(description='輸入干擾：柏林噪聲', value=False, indent=False)
noise_comp = widgets.Checkbox(description='輸入干擾：壓縮失真', value=False, indent=False)

# --- 控制參數（投影片 P.71 的範圍） ----------------------------------------
speed_gain = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.0, description='speed gain', **S)
steer_gain = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.2, description='steering gain', **S)
steer_kd = widgets.FloatSlider(min=0.0, max=0.5, step=0.001, value=0.0, description='steering kd',
                               readout_format='.3f', **S)
steer_bias = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.0, description='steering bias', **S)

# --- 即時讀數（投影片 P.73） -----------------------------------------------
x_r = widgets.FloatSlider(min=-1.0, max=1.0, description='x', **R)
y_r = widgets.FloatSlider(min=-1.0, max=1.0, description='y', **R)
angle_r = widgets.FloatSlider(min=-1.6, max=1.6, description='angle (rad)', **R)
steer_r = widgets.FloatSlider(min=-1.0, max=1.0, description='steering', **R)
left_r = widgets.FloatProgress(min=0.0, max=1.0, description='left motor', **S)
right_r = widgets.FloatProgress(min=0.0, max=1.0, description='right motor', **S)
stats = widgets.HTML()

# --- 運轉 ---------------------------------------------------------------
allow_move = widgets.Checkbox(description='允許移動（送馬達）', value=False, indent=False)
record = widgets.Checkbox(description='錄製 CSV', value=False, indent=False)
start_btn = widgets.Button(description='開始推論', button_style='success')
stop_btn = widgets.Button(description='停止', button_style='danger')
image_w = widgets.Image(format='jpeg', width=336, height=336)

ui = widgets.VBox([
    widgets.HBox([model_dd, refresh_btn]), model_info,
    widgets.HBox([
        widgets.VBox([image_w, widgets.HBox([noise_perlin, noise_comp])]),
        widgets.VBox([widgets.HTML('<b>控制參數</b>'), speed_gain, steer_gain, steer_kd, steer_bias,
                      widgets.HTML('<b>即時讀數</b>'), x_r, y_r, angle_r, steer_r, left_r, right_r, stats]),
    ]),
    widgets.HBox([start_btn, stop_btn, allow_move, record]),
])
display(ui)

## 4. 推論 / 控制迴圈

In [ ]:
state = {'running': False, 'thread': None, 'model': None, 'pending': None, 'error': None,
         'csv': None, 'writer': None}

def describe(m):
    meta = m.meta or {}
    parts = ['<b>%s</b>' % m.describe()]
    if meta.get('label'): parts.append(meta['label'])
    if meta.get('params'): parts.append('params %.2fM' % (meta['params'] / 1e6))
    if meta.get('macs'): parts.append('MACs %.2fG' % (meta['macs'] / 1e9))
    if meta.get('best_val_loss') is not None: parts.append('val loss %.4f' % meta['best_val_loss'])
    return ' | '.join(parts)

def halt():
    robot.stop()
    left_r.value = right_r.value = 0.0

def set_recording(on):
    if on and state['csv'] is None:
        os.makedirs(LOG_DIR, exist_ok=True)
        path = os.path.join(LOG_DIR, 'drive_%s.csv' % datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
        state['csv'] = open(path, 'w', newline='')
        state['writer'] = csv.writer(state['csv'])
        state['writer'].writerow(['t', 'model', 'moving', 'perlin', 'compression', 'x', 'y', 'angle',
                                  'p', 'd', 'steering', 'speed', 'left', 'right', 'latency_ms',
                                  'kp', 'kd', 'bias'])
        print('recording ->', path)
    elif not on and state['csv'] is not None:
        state['csv'].close()
        state['csv'] = state['writer'] = None

def loop():
    fps_t, fps_n, fps = time.time(), 0, 0.0
    lat_avg = None
    try:
        while state['running']:
            # 換模型在迴圈執行緒內做（CUDA / TensorRT context 一致）。
            # 換的時候停車並取消「允許移動」，確認新模型的預測點合理後再手動勾選。
            if state['pending'] is not None:
                path, state['pending'] = state['pending'], None
                allow_move.value = False
                halt()
                model_info.value = '載入中… %s' % os.path.basename(path)
                state['model'] = None
                state['model'] = load_model(path, device)
                controller.reset()
                lat_avg = None
                model_info.value = describe(state['model'])
            m = state['model']
            frame = camera.read()
            if frame is None or m is None:
                time.sleep(0.02)
                continue
            frame = cam_mod.degrade(frame, noise_perlin.value, noise_comp.value)

            t0 = time.time()
            out = m(cam_mod.preprocess(frame, device, m.dtype))
            xy = out.detach().float().cpu().numpy().flatten()
            if device.type == 'cuda':
                torch.cuda.synchronize()
            lat = (time.time() - t0) * 1000.0
            lat_avg = lat if lat_avg is None else 0.9 * lat_avg + 0.1 * lat

            controller.speed, controller.kp = speed_gain.value, steer_gain.value
            controller.kd, controller.bias = steer_kd.value, steer_bias.value
            x, y = model_output_to_xy(xy)
            left, right = controller.update(x, y)
            c = controller.last
            if allow_move.value:
                robot.left_motor.value, robot.right_motor.value = left, right
            else:
                robot.stop()

            x_r.value, y_r.value = x, y
            angle_r.value, steer_r.value = c['angle'], max(-1.0, min(1.0, c['steering']))
            left_r.value, right_r.value = left, right
            image_w.value = cam_mod.to_jpeg(cam_mod.draw_overlay(frame, xy, c['steering']))

            fps_n += 1
            if time.time() - fps_t >= 1.0:
                fps, fps_t, fps_n = fps_n / (time.time() - fps_t), time.time(), 0
            stats.value = ('FPS <b>%.1f</b> | 推論 <b>%.1f ms</b>（平均 %.1f）| P %.3f D %.3f | %s'
                           % (fps, lat, lat_avg, c['p'], c['d'],
                              '<span style="color:red">移動中</span>' if allow_move.value else '未送馬達'))
            set_recording(record.value)
            if state['writer'] is not None:
                state['writer'].writerow(['%.3f' % time.time(), os.path.basename(m.path), int(allow_move.value),
                                          int(noise_perlin.value), int(noise_comp.value)]
                                         + ['%.4f' % c[k] for k in ('x', 'y', 'angle', 'p', 'd', 'steering',
                                                                    'speed', 'left', 'right')]
                                         + ['%.2f' % lat, controller.kp, controller.kd, controller.bias])
    except Exception as e:
        state['error'] = e
        model_info.value = '<span style="color:red">錯誤：%r</span>' % (e,)
    finally:
        halt()
        set_recording(False)
        state['running'] = False

def on_start(_=None):
    if state['running']:
        return
    if state['model'] is None and model_dd.value:
        state['pending'] = model_dd.value
    state['running'], state['error'] = True, None
    state['thread'] = threading.Thread(target=loop, daemon=True)
    state['thread'].start()

def on_stop(_=None):
    allow_move.value = False
    state['running'] = False
    halt()

def on_model_change(change):
    if change['new']:
        state['pending'] = change['new']

def on_refresh(_=None):
    model_dd.options = list_models(MODELS_DIR)

start_btn.on_click(on_start)
stop_btn.on_click(on_stop)
refresh_btn.on_click(on_refresh)
model_dd.observe(on_model_change, names='value')

## 5. 推論速度比較（報告用）

在**車子不動**的情況下，對 `models/` 內每個模型用同一張畫面量 latency，條件相同才能比較（P.80）。先按「停止」再執行。

In [ ]:
def benchmark(n=100, warmup=20):
    assert not state['running'], '先按「停止」'
    frame = camera.read()
    rows = []
    for label, path in list_models(MODELS_DIR):
        m = load_model(path, device)
        x = cam_mod.preprocess(frame, device, m.dtype)
        for _ in range(warmup):
            m(x)
        if device.type == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n):
            m(x)
        if device.type == 'cuda': torch.cuda.synchronize()
        ms = (time.time() - t0) * 1000.0 / n
        rows.append((ms, label, m.kind))
        del m
    for ms, label, kind in sorted(rows):
        print('%8.2f ms  %6.1f FPS  [%s]  %s' % (ms, 1000.0 / ms, kind, label))
    return rows

# benchmark()

## 6. 結束：停車並釋放相機

In [ ]:
on_stop()
time.sleep(0.2)
camera.stop()